# Assignment 4: Retrieval-Augmented Generation (RAG)

In this assignment you will build a RAG pipeline using **LangChain** to answer
medical yes/no questions from the **PubMedQA** dataset. You will:

1. Load PubMedQA, filter for yes/no items, and build a documents/questions split.
2. Configure a small open-source language model via `HuggingFacePipeline`.
3. Set up an embedding model, chunk the documents, and store them in a
   **Chroma** vector store.
4. Wire a full RAG pipeline (you may pick the agent-middleware path or the
   LCEL path).
5. Evaluate the pipeline against an LM-only baseline and inspect retrieval
   quality.

Cells marked **⚙️** are configuration/scaffolding (already filled in for you).
Cells marked **🎓** are graded — you must implement the missing pieces.

## Preliminaries

In [ ]:
%pip install -q \
    langchain langchain-community langchain-huggingface langchain-core \
    langchain-chroma langchain-text-splitters \
    sentence-transformers chromadb \
    pandas scikit-learn

In [ ]:
import os
import json
import urllib.request
from pathlib import Path

import torch
import pandas as pd
import numpy as np

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

SEED = 101
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)
PUBMEDQA_PATH = DATA_DIR / "ori_pqal.json"
CHROMA_DIR = "./chroma_pubmedqa"


## Part 1: PubMedQA dataset

### ⚙️ Task 1.1 — Download and inspect

We download the original PubMedQA labelled set (`ori_pqal.json`), keep only
items whose `final_decision` is `yes` or `no`, and build two DataFrames.
You don't need to modify anything here.

In [ ]:
PUBMEDQA_URL = (
    "https://raw.githubusercontent.com/pubmedqa/pubmedqa/"
    "refs/heads/master/data/ori_pqal.json"
)

if not PUBMEDQA_PATH.exists():
    print(f"Downloading {PUBMEDQA_URL} ...")
    urllib.request.urlretrieve(PUBMEDQA_URL, PUBMEDQA_PATH)
print(f"PubMedQA file: {PUBMEDQA_PATH} ({PUBMEDQA_PATH.stat().st_size:,} bytes)")


In [ ]:
tmp_data = pd.read_json(PUBMEDQA_PATH).T
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({
    "abstract": tmp_data.apply(
        lambda row: " ".join(row.CONTEXTS + [row.LONG_ANSWER]), axis=1),
    "year": tmp_data.YEAR})

questions = pd.DataFrame({
    "question": tmp_data.QUESTION,
    "year": tmp_data.YEAR,
    "gold_label": tmp_data.final_decision,
    "gold_context": tmp_data.LONG_ANSWER,
    "gold_document_id": documents.index})

print(f"documents: {documents.shape}")
print(f"questions: {questions.shape}")
documents.head(2)


In [ ]:
questions.head(3)

## Part 2: Language model

### ⚙️ Task 2.1 — Configure a HuggingFace LM

Pick a small instruction-tuned model. Suggestions:

- `Qwen/Qwen2.5-0.5B-Instruct` (default — small, open, no gating)
- `HuggingFaceTB/SmolLM2-360M-Instruct` (tiny, weakest)
- `Qwen/Qwen2.5-1.5B-Instruct` (better, slower)

Gated models like Llama 3.2 require a HuggingFace token with "Read access to
contents of all public gated repos you can access" — you'd then call
`huggingface-cli login` first.

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

LM_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# Build the transformers pipeline manually so we can pass DEVICE='mps'.
# (HuggingFacePipeline.from_model_id only accepts integer CUDA ids and
# crashes on string devices like 'mps'.)
tokenizer = AutoTokenizer.from_pretrained(LM_MODEL_ID)
hf_model = AutoModelForCausalLM.from_pretrained(LM_MODEL_ID).to(DEVICE)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

text_gen = hf_pipeline(
    "text-generation",
    model=hf_model,
    tokenizer=tokenizer,
    max_new_tokens=64,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.pad_token_id,
)

llm = HuggingFacePipeline(pipeline=text_gen)

# Quick sanity check.
print(llm.invoke("Q: What is the capital of France?\nA:"))


## Part 3: Building the retrieval index

### 🎓 Task 3.1 — Embedding model

Define a `HuggingFaceEmbeddings` instance. Verify the embedding dimension
by calling `embed_query()` on a sample sentence and printing the length of
the returned vector.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

# TODO: instantiate `embedding_model` with HuggingFaceEmbeddings.
# Hint: you can pass model_kwargs={"device": ...} and
# encode_kwargs={"normalize_embeddings": True}.
embedding_model = ...
raise NotImplementedError("Task 3.1: define embedding_model")


In [ ]:
# Sanity check.
sample_vec = embedding_model.embed_query("What is programmed cell death?")
print(f"Embedding dim: {len(sample_vec)}")
print(f"First 5 values: {sample_vec[:5]}")


### ⚙️ Task 3.2 — Chunking

Use `RecursiveCharacterTextSplitter` to split each abstract into chunks. The
metadata for each chunk should record the **document id** so we can later
check whether the retriever returned the gold abstract.

**Reflection (write your answer in a markdown cell below if you wish):**
How do `chunk_size` and `chunk_overlap` affect retrieval and answer quality?

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(
    texts=documents.abstract.tolist(),
    metadatas=metadatas,
)
texts = text_splitter.split_documents(texts)
print(f"Number of chunks: {len(texts)}")
print(f"First chunk preview: {texts[0].page_content[:200]}...")
print(f"Metadata: {texts[0].metadata}")


### 🎓 Task 3.3 — Vector store

Build a Chroma vector store from the chunks. Use **cosine similarity**
(`collection_metadata={"hnsw:space": "cosine"}`).

Then run a sanity-check `similarity_search_with_score` query and inspect the
returned documents.

In [ ]:
from langchain_chroma import Chroma

# TODO: build `vector_store` from `texts` using `embedding_model` and
# cosine similarity. Persist to CHROMA_DIR.
vector_store = ...
raise NotImplementedError("Task 3.3: build vector_store")


In [ ]:
results = vector_store.similarity_search_with_score(
    "What is programmed cell death?", k=3)
for res, score in results:
    print(f"* [SIM={score:.3f}] doc_id={res.metadata.get('id')}")
    print(f"  {res.page_content[:200]}...")
    print()


## Part 4: RAG pipeline

### 🎓 Task 4.1 — Define the full pipeline

Pick **one** of the two paths described in the assignment:

**Option A (agent-based)**: implement a `RetrieveDocumentsMiddleware` that
extends `AgentState` with a `context: list[Document]` field, retrieves
documents in `before_model`, and returns the augmented message + retrieved
context. Then build the agent with `create_agent(model, tools=[], middleware=[...])`.

**Option B (LCEL)**: define a `ChatPromptTemplate`, a generation chain
`prompt | llm | StrOutputParser()`, and a `RunnableParallel` that returns
both the retrieved context and the answer.

Either way, the pipeline should:

1. Take a question (string) as input.
2. Retrieve top-k similar chunks from the vector store.
3. Stuff them into a prompt that asks for a yes/no answer.
4. Return both the model's answer **and** the retrieved documents.

You should also create an **LM-only baseline** chain that uses the same
prompt template *without* retrieved context, for comparison in Task 5.1.

In [ ]:
# TODO: define `rag_chain` (returns dict with at least 'answer' and
# 'retrieved_docs' keys) and `lm_only_chain` (returns string answer).
#
# You will need at least:
#   - retriever = vector_store.as_retriever(search_kwargs={"k": 3})
#   - a ChatPromptTemplate that asks for yes/no
#   - the LLM you built in Task 2.1
#   - StrOutputParser to turn LM output into a string
#
# For the LCEL path, the structure is roughly:
#   from langchain_core.prompts import ChatPromptTemplate
#   from langchain_core.runnables import RunnablePassthrough, RunnableParallel
#   from langchain_core.output_parsers import StrOutputParser
#
#   rag_chain = RunnableParallel({...}).assign(answer=prompt | llm | parser)

raise NotImplementedError("Task 4.1: define rag_chain and lm_only_chain")


In [ ]:
# Smoke test on one question.
sample_q = questions.iloc[0]
print(f"Q: {sample_q['question']}")
print(f"Gold: {sample_q['gold_label']}")
print()

result = rag_chain.invoke(sample_q["question"])
print(f"RAG answer: {result['answer']!r}")
# If you used the agent-middleware path, the keys may be different — adapt below.
print(f"Retrieved doc ids: {[d.metadata.get('id') for d in result['retrieved_docs']]}")
print(f"Gold doc id: {sample_q['gold_document_id']}")
print()
print(f"LM-only answer: {lm_only_chain.invoke(sample_q['question'])!r}")


## Part 5: Evaluation

### 🎓 Task 5.1 — High-level evaluation

Run both pipelines on a subset of the question set and report:

- **valid-answer rate** (fraction of rows where the model produced a parseable
  yes/no)
- **accuracy** over valid rows
- **macro F1** over valid rows

Compare RAG vs. LM-only — does retrieval help?

In [ ]:
import re
from sklearn.metrics import accuracy_score, f1_score, classification_report

N_EVAL = 50  # increase to evaluate on more questions
eval_set = questions.iloc[:N_EVAL].copy()


def parse_yes_no(text):
    '''Extract 'yes' or 'no' from model output. Returns None if neither found.'''
    if text is None:
        return None
    t = text.strip().lower()
    head = t[:40]
    m_yes = re.search(r"\byes\b", head)
    m_no = re.search(r"\bno\b", head)
    if m_yes and (not m_no or m_yes.start() < m_no.start()):
        return "yes"
    if m_no and (not m_yes or m_no.start() < m_yes.start()):
        return "no"
    return None


In [ ]:
# TODO: run rag_chain and lm_only_chain over `eval_set["question"]`.
# Collect the raw output and the parsed yes/no into rag_results / lm_results.
# Time both runs so you can report seconds-per-question.

raise NotImplementedError("Task 5.1: run both pipelines and collect results")


In [ ]:
# TODO: compute and print accuracy, macro-F1, valid rate for both pipelines.
# Use sklearn's accuracy_score and f1_score(..., labels=["yes","no"], average="macro").

raise NotImplementedError("Task 5.1: report metrics")


### 🎓 Task 5.2 — Detailed inspection

Two questions:

1. **Recall**: for each question, is `gold_document_id` among the top-k
   retrieved chunk ids?
2. **Qualitative**: pick a few correct/incorrect cases and inspect the
   retrieved chunks vs. the gold context to understand failures.

In [ ]:
# TODO: compute recall@k and recall@1 of the retriever using
# `gold_document_id` as ground truth.

raise NotImplementedError("Task 5.2: compute retrieval recall")


In [ ]:
# TODO: print a few qualitative examples — at least one correct and one
# wrong. Show: question, gold label, RAG answer, retrieved doc ids, gold doc id,
# and the gold context.

raise NotImplementedError("Task 5.2: qualitative inspection")


## Wrap-up

Once you have results, reflect on:

- Does RAG beat the LM-only baseline? By how much?
- When the retriever finds the gold document, does the LM answer correctly?
- When the retriever misses the gold document, what kind of chunk does it
  return instead?
- What would you change to improve the pipeline (better embedding model,
  larger LM, hybrid retrieval, prompt engineering, ...)?